In [25]:
import requests
import polars as pl
from datetime import datetime
from collections import defaultdict
import pandas as pd
from ts_tariffs.utils import Block
from ts_tariffs.billing import Bill
from ts_tariffs.meters import MeterData
from ts_tariffs.ts_utils import SampleRate, DateWindow, TimeWindow
from ts_tariffs.tariffs import tariffs_map

elecTariff = "3354537"
zip_code = "55042"
building = pl.read_csv("load_profiles/CA/128427-0.csv")

app_id = "3df8e135-968d-4399-9879-2a1c6a3de30c"
app_key = "e51974c7-996b-4698-9628-71950d223364"

url = "https://api.genability.com/rest/public/territories"
params = {
    "masterTariffId": elecTariff,
    "zipCode": zip_code
}

response = requests.get(url, auth=(app_id, app_key), params=params).json()
territoryId = response["results"][0]["territoryId"] if len(response["results"])>0 else 0

url = "https://api.genability.com/rest/v1/ondemand/calculate"
params = {
    "masterTariffId": elecTariff,
    "zipCode": zip_code,
    "fromDateTime": building["timestamp"].first(),
    "toDateTime": building["timestamp"].last(),
    "groupBy": "YEAR",
    "propertyInputs" : [{
        "keyName": "consumption",
        "unit": "kWh",
        "fromDateTime": building["timestamp"].first(),
        "duration": 900000, # 15 mins
        "dataSeries": building["electricity.total"].to_list()
    }]
}
if territoryId:
    params["propertyInputs"].append({"keyName": "territoryId", "dataValue": territoryId})

response = requests.post(url, auth=(app_id, app_key), json=params)
data = response.json()["results"][0]
name = data["tariffName"]
costs_breakdown = pl.from_dicts(response.json()["results"][0]["items"])
cols = ["rateName", "rateAmount", "itemQuantity","cost","period"] if "period" in costs_breakdown.columns else ["rateName", "itemQuantity", "rateAmount","cost"]
costs_breakdown = costs_breakdown.select(cols).sort("rateName")

In [26]:
url = "https://api.genability.com/rest/public/tariffs"

params = {
    "masterTariffId": elecTariff,
    "zipCode": zip_code,
    "effectiveOn": datetime.now().strftime("%Y-%m-%d"),
    "fromDateTime": building["timestamp"].first(),
    "toDateTime": building["timestamp"].last(),
    "populateRates": True
}

response = requests.get(url, auth=(app_id, app_key), params=params)
rates = [r for r in response.json()["results"][0]["rates"] if r["rateName"] in costs_breakdown["rateName"] and \
            (any(c["rateAmount"]!=0 and c["itemQuantity"]!=0 for c in costs_breakdown.filter(pl.col("rateName")==r["rateName"]).to_dicts()) or \
             any(b["rateAmount"]!=0 for b in r["rateBands"])) and \
            r.get("territory",{"territoryId":territoryId})["territoryId"]==territoryId]

In [27]:
# Update the logic with rate determinant, seasonal, time of use, and block handling
rows = []
for rate in rates:
    rate_name = rate["rateName"]
    eff_date = rate["fromDateTime"].split("T")[0]
    
    # Rate Determinant logic
    if rate.get("chargeType") == "FIXED_PRICE" and \
        any(c>=0 for c in costs_breakdown.filter(pl.col("rateName")==rate["rateName"])["rateAmount"]) and \
        any(q==12 or q==13 for q in costs_breakdown.filter(pl.col("rateName")==rate["rateName"])["itemQuantity"]):
        rate_determinant = "per month"
    elif rate.get("chargeType") == "FIXED_PRICE" and any(q<12 and q==round(q) for q in costs_breakdown.filter(pl.col("rateName")==rate["rateName"])["itemQuantity"]):
        rate_determinant = "per year"
    elif rate.get("chargeType") == "QUANTITY" and (rate["rateBands"] or [{1:""}])[0].get("rateUnit")=="PERCENTAGE":
        rate_determinant = "percent"
    elif rate.get("chargeType") == "DEMAND_BASED":
        rate_determinant = "per kw" # Always assuming 30min peak
    else:
        rate_determinant = "per kwh"

    # Season Logic
    season = ""
    if rate.get("season",rate.get("timeOfUse",{}).get("season")):
        s = rate.get("season",rate.get("timeOfUse",{}).get("season"))
        season = f'{s["seasonFromMonth"]:02d}/{s["seasonFromDay"]:02d}-{s["seasonToMonth"]:02d}/{s["seasonToDay"]:02d}'
    
    # Time of Use Logic
    tou_type = ""
    tou = ""
    if "timeOfUse" in rate:
        tou=[]
        for p in rate["timeOfUse"]["touPeriods"]:
            tou.append(([p["fromDayOfWeek"]+1,p["toDayOfWeek"]+1 if p["toDayOfWeek"]<6 else p["toDayOfWeek"]+2],[p["fromHour"], p["toHour"]]))
        tou = str(tou)
        tou_type = rate["timeOfUse"].get("touType", "OFF_PEAK")
    
    # Block Logic
    bands = rate.get("rateBands", [])
    has_cons_limits = any(b.get("hasConsumptionLimit") for b in bands)
    if not has_cons_limits or len(bands) == 1:
        rows.append([name, rate_name, eff_date, "", rate_determinant, "", "", season, tou, tou_type])
    else:
        limits = [b.get("consumptionUpperLimit") for b in bands]
        prev_limit = None
        for i, limit in enumerate(limits):
            if limit == prev_limit:
                continue  # skip dupes
            start = "" if i == 0 else prev_limit
            end = limit if limit is not None else ""
            rows.append([name, rate_name, eff_date, "", rate_determinant, start, end, season, tou, tou_type])
            prev_limit = limit

df = pl.DataFrame(rows, schema=[
    "tariff","rateName", "EffDate", "Rate", "Rate Determinant",
    "Start", "End", "Season", "tou", "period"
])



C:\Users\al.qarooni\AppData\Local\Temp\ipykernel_33988\72686493.py:53: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(rows, schema=[


In [28]:
# group df by rateName
df_grouped = df.group_by("rateName","Start","End","Rate Determinant",maintain_order=True).agg(pl.len().alias("df_count"))
cb_grouped = costs_breakdown.group_by("rateName",maintain_order=True).agg([
    pl.len().alias("cb_count"),
    (pl.col("rateAmount") * pl.col("itemQuantity")).sum().alias("weighted_sum"),
    (pl.col("cost").sum()/building["electricity.total"].sum()).alias("sum_%"), # Converting percentage based to per kwh - idea is to use base calc then scale based on consumption of new buildings
    pl.col("itemQuantity").sum().alias("total_qty"),
    pl.col("rateAmount").alias("rate_list"),  # keep as list for ordered match
    (pl.col("cost")/building["electricity.total"].sum()).alias("rate_list_%")  # keep as list for ordered match
])

# join stats
summary = df_grouped.join(cb_grouped, on="rateName", how="left")

# result mapping: rateName -> list of final rates
rate_map = {}
for row in summary.iter_rows(named=True):
    name = row["rateName"]
    determinant = row["Rate Determinant"]
    dcount = row["df_count"]
    ccount = row["cb_count"]
    factor = row["total_qty"] if determinant=="per year" else 1

    if ccount == 1 and dcount > 1:
        # broadcast single rate
        val = factor * row["rate_list"][0] if determinant!="percent" else row["rate_list_%"]
        rate_map[name] = [val] * dcount

    elif dcount == 1:
        # assign weighted avg
        avg = factor * row["weighted_sum"] / row["total_qty"] if determinant!="percent" and row["total_qty"] else row["sum_%"] if determinant=="percent" else 0
        rate_map[name] = [avg]

    elif ccount == dcount:
        # respect order
        rate_map[name] = [r*factor for r in row["rate_list"]] if determinant!="percent" else row["rate_list_%"]
    
    else:
        # mismatch, fallback: NaNs
        rate_map[name] = [None] * dcount

final_rates = []
group_counts = {}

for row in df.iter_rows(named=True):
    name = row["rateName"]
    from_tariff = [band["rateAmount"] * (-1 if band["isCredit"] else 1) for r in rates if r["rateName"]==name for band in r["rateBands"] if band["rateUnit"]!="PERCENTAGE"]
    if name not in group_counts:
        group_counts[name] = 0
    idx = group_counts[name]
    if any([c!=0 for c in from_tariff]):
        val_list=from_tariff
    else:
        val_list = rate_map.get(name, []) if "per kw" != row["Rate Determinant"] else [r/12 for r in rate_map.get(name, [])]
    val = val_list[idx] if idx < len(val_list) else None
    final_rates.append(val)
    group_counts[name] += 1

df = df.with_columns(pl.Series("Rate", final_rates)).unique().sort("rateName")

In [29]:
df.unique().sort("rateName")

tariff,rateName,EffDate,Rate,Rate Determinant,Start,End,Season,tou,period
str,str,str,f64,str,str,str,str,str,str
"""Residential Electric Vehicle S…","""Affordability Surcharge""","""2024-01-01""",1.956813,"""per month""","""""","""""","""""","""""",""""""
"""Residential Electric Vehicle S…","""Conservation Improvement Progr…","""2024-01-01""",-0.000389,"""per kwh""","""""","""""","""""","""""",""""""
"""Residential Electric Vehicle S…","""Customer Charge""","""2024-01-01""",42.5,"""per month""","""""","""""","""""","""""",""""""
"""Residential Electric Vehicle S…","""Excess Energy Charge - Summer …","""2024-01-01""",0.0,"""per kwh""","""""","""340""","""06/01-09/30""","""[([1, 5], [9, 21])]""","""ON_PEAK"""
"""Residential Electric Vehicle S…","""Excess Energy Charge - Summer …","""2024-01-01""",0.25879,"""per kwh""","""340""","""""","""06/01-09/30""","""[([1, 5], [9, 21])]""","""ON_PEAK"""
…,…,…,…,…,…,…,…,…,…
"""Residential Electric Vehicle S…","""Excess Energy Charge - Winter …","""2024-01-01""",0.21408,"""per kwh""","""340""","""""","""10/01-05/31""","""[([1, 5], [9, 21])]""","""ON_PEAK"""
"""Residential Electric Vehicle S…","""Fuel Cost Charge""","""2024-01-01""",0.027345,"""per kwh""","""""","""""","""""","""""",""""""
"""Residential Electric Vehicle S…","""Renewable Development Fund""","""2024-01-01""",0.001178,"""per kwh""","""""","""""","""""","""""",""""""


In [30]:
from collections import defaultdict

# Drop rows with null Rate
df = df.filter(pl.col("Rate").is_not_null())

# Prepare load data with month, hour, and date columns
building_filter_mast = building.with_columns([
    pl.col("timestamp").str.to_datetime().alias("timestamp")
])
building_filter_mast = building_filter_mast.with_columns([
    pl.col("timestamp").dt.month().alias("month"),
    pl.col("timestamp").dt.hour().alias("hour"),
    pl.col("timestamp").dt.date().alias("date"),
    pl.col("timestamp").dt.weekday().alias("weekday")
])

# Initialize charges
total_charges = defaultdict(float)

# Convert rate table to row-wise dict for iteration
for row in df.iter_rows(named=True):
    name = row["rateName"]
    rate = row["Rate"]
    determinant = str(row["Rate Determinant"]).lower()
    season = row["Season"]
    tou = row["tou"]
    start = float(row["Start"]) if row["Start"] else 0.0
    end = float(row["End"]) if row["End"] else float("inf")

    # Get season months
    if isinstance(season, str) and "/" in season:
        mo1, day1 = map(int, season.split("-")[0].split("/"))
        mo2, day2 = map(int, season.split("-")[1].split("/"))
        season_months = list(range(mo1, mo2 + 1)) if mo1 <= mo2 else list(range(mo1, 13)) + list(range(1, mo2 + 1))
    else:
        season_months = list(range(1, 13))

    # Filter load for season
    building_filter = building_filter_mast.filter(pl.col("month").is_in(season_months))

    # Filter for TOU if applicable
    if tou:
        temp = []
        tou = eval(tou)
        for t_d, t_h in tou:
            start_day, end_day = t_d
            start_time, end_time = t_h
            if start_time < end_time:
                temp.append(building_filter.filter((pl.col("hour") >= start_time) & (pl.col("hour") < end_time) &
                                                    (pl.col("weekday")>= start_day) & (pl.col("weekday") < end_day)))
            else:
                temp.append(building_filter.filter(((pl.col("hour") >= start_time) | (pl.col("hour") < end_time)) &
                                                    (pl.col("weekday")>= start_day) & (pl.col("weekday") < end_day)))
        building_filter = pl.concat(temp)
    
    if "kwh" in determinant:
        # Compute monthly totals
        building_filter_monthly_total = (
            building_filter.group_by("month")
              .agg(pl.col("electricity.total").sum().alias("month_total"))
        )
        building_filter = building_filter.join(building_filter_monthly_total, on="month")
        
        # Consumption limit handling
        building_filter = building_filter.with_columns([
            pl.when(pl.col("month_total") > start)
            .then(
                pl.when(pl.col("month_total") > end)
                    .then(end - start)
                    .otherwise(pl.col("month_total") - start)
            )
            .otherwise(0)
            .alias("adjusted_kwh_factor")
        ])

        building_filter = building_filter.with_columns([
            (pl.col("electricity.total") * pl.col("adjusted_kwh_factor") / pl.col("month_total"))
            .fill_nan(0)
            .alias("adjusted_kwh")
        ])

        kwh = building_filter["adjusted_kwh"].sum()
        charge = rate * kwh
        total_charges[name] += charge

    elif "kw" in determinant:
        building_filter = (
            building_filter.group_by("month").agg(
               (pl.col("electricity.total").rolling_sum(window_size=2).max()/0.5).alias("30min_peak_demand")
            )
        )
        
        kw = building_filter["30min_peak_demand"].sum()
        charge = rate * kw
        total_charges[name] += charge

    elif "month" in determinant or "bill" in determinant:
        months = building_filter["month"].n_unique()
        total_charges[name] += rate * months

    elif "day" in determinant:
        days = building_filter["date"].n_unique()
        total_charges[name] += rate * days

    elif "year" in determinant:
        total_charges[name] += rate
    
    elif "percent" in determinant:
        total_charges[name] += building_filter["electricity.total"].sum() * rate

# Sum all charges
annual_total_bill = sum(total_charges.values())
print(annual_total_bill)
total_charges = dict(total_charges)
total_charges


850.36854323515


{'Affordability Surcharge': 23.48176176,
 'Conservation Improvement Program Adjustment': -3.6197286350000004,
 'Customer Charge': 510.0,
 'Excess Energy Charge - Summer On-Peak': 0.0,
 'Excess Energy Charge - Winter On-Peak': 0.0,
 'Fuel Cost Charge': 254.44803345405,
 'Renewable Development Fund': 10.9570767668,
 'Renewable Energy Standard Adjustment': 9.60801765,
 'Transmission Cost Recovery Charge': 45.4933822393}

In [31]:
print(costs_breakdown["cost"].sum())
costs_breakdown = costs_breakdown.group_by("rateName").agg([pl.col("rateAmount").mean(), pl.col("cost").sum(),pl.col("itemQuantity").sum().round(3)]).sort(by="rateName")
costs_breakdown.select(["rateName","cost"]).to_dicts()

873.82853113


[{'rateName': 'Affordability Surcharge', 'cost': 25.43857527},
 {'rateName': 'Conservation Improvement Program Adjustment',
  'cost': -3.61964111},
 {'rateName': 'Customer Charge', 'cost': 509.97143817},
 {'rateName': 'Excess Energy Charge - Summer Off-Peak', 'cost': 0.0},
 {'rateName': 'Excess Energy Charge - Summer On-Peak', 'cost': 21.53914782},
 {'rateName': 'Excess Energy Charge - Winter Off-Peak', 'cost': 0.0},
 {'rateName': 'Excess Energy Charge - Winter On-Peak', 'cost': 0.0},
 {'rateName': 'Fuel Cost Charge', 'cost': 254.44183799},
 {'rateName': 'Mercury Cost Recovery', 'cost': 0.0},
 {'rateName': 'Renewable Development Fund', 'cost': 10.95684462},
 {'rateName': 'Renewable Energy Standard Adjustment', 'cost': 9.60801765},
 {'rateName': 'State Energy Policy Rate', 'cost': 0.0},
 {'rateName': 'Transmission Cost Recovery Charge', 'cost': 45.49231072}]

In [186]:
# Accumulators for tiers
global_blocks   = defaultdict(lambda: {"blocks": [], "rates": [], "labels": []})
seasonal_blocks = defaultdict(lambda: {"blocks": [], "rates": [], "labels": []})

charges = []

for row in df.iter_rows(named=True):
    if not row["Rate"]:
        continue
    if row.get("Location"):
        continue

    comp       = row["rateName"]
    tou_raw    = row.get("tou")
    rate       = float(row["Rate"])
    time_bins  = ""
    if tou_raw:
        time_bins  = eval(tou_raw)
        if len(time_bins)<2:
            rate = [0.0, rate]
        elif time_bins[0] > time_bins[1]:
            rate = [rate, 0.0, rate]
            time_bins = [time_bins[1], time_bins[0]]
        else:
            rate = [0.0, rate, 0.0]
        time_bins+=[24]
    det        = row.get("Rate Determinant") or ""
    unit       = det.split("per ")[1].lower() if "per " in det else "therm"
    start      = row.get("Start")
    end        = row.get("End")
    season_txt = row.get("Season")

    # 1) seasonal block rows
    if season_txt and (start or end):
        key = (comp, unit, season_txt)
        s = float(start) if start else 0.0
        e = float(end)   if end   else float("inf")
        sb = seasonal_blocks[key]
        sb["blocks"].append(Block(min=s, max=e))
        sb["rates"].append(rate)
        sb["labels"].append("")
        if time_bins:
            sb["time_bins"] = [0]+time_bins
        continue

    # 2) seasonal single-rates (no block thresholds)
    if season_txt and not (start or end):
        sd, ed = (s.strip() for s in season_txt.split("-"))
        ms, ds = map(int, sd.split("/"))
        me, de = map(int, ed.split("/"))

        windows = []
        if (me, de) < (ms, ds):
            windows = [
                (f"2025-{ms:02d}-{ds:02d}", "2025-12-31"),
                ("2025-01-01", f"2025-{me:02d}-{de:02d}")
            ]
        else:
            windows = [(f"2025-{ms:02d}-{ds:02d}", f"2025-{me:02d}-{de:02d}")]

        for ws, we in windows:
            if tou_raw:
                charges.append({
                    "name": f"{comp}_{unit}_{ws[-5:]}_to_{we[-5:]}_tou",
                    "charge_type": "TouTariff",
                    "tou":{
                        "bin_rates": rate,
                        "bin_labels": ["", "", ""],
                        "time_bins": time_bins,
                    },
                    "consumption_unit": unit,
                    "rate_unit": f"dollars / {unit}",
                    "sample_rate": None,
                    "adjustment_factor": 1.0,
                    "__season_window__": (ws, we)
                })
            else:
                charges.append({
                    "name": f"{comp}_{unit}_{ws[-5:]}_to_{we[-5:]}_single",
                    "charge_type": "SingleRateTariff",
                    "rate": rate,
                    "consumption_unit": unit,
                    "rate_unit": f"dollars / {unit}",
                    "sample_rate": None,
                    "adjustment_factor": 1.0,
                    "__season_window__": (ws, we)
                })
        continue

    # 3) non‐seasonal block rows
    if start or end:
        key = (comp, unit)
        s = float(start) if start else 0.0
        e = float(end)   if end   else float("inf")
        gb = global_blocks[key]
        gb["blocks"].append(Block(min=s, max=e))
        gb["rates"].append(rate)
        gb["labels"].append("")
        if time_bins:
            gb["time_bins"] = [0]+time_bins
        continue

    # 4) connection charges
    if unit in {"day", "month", "year", "bill"}:
        if unit =="bill": 
            unit = "month"
        charges.append({
            "name": f"{comp}_{unit}_connection",
            "charge_type": "ConnectionTariff",
            "rate": rate,
            "consumption_unit": unit,
            "rate_unit": f"dollars / {unit}",
            "frequency_applied": unit,
            "sample_rate": None,
            "adjustment_factor": 1.0,
        })
        continue

    # 5) flat TOU (non-seasonal, non-block)
    if tou_raw:
        charges.append({
            "name": f"{comp}_{unit}_tou_single",
            "charge_type": "TouTariff",
            "tou": {
                "bin_rates": rate,
                "bin_labels": ["", "", ""],
                "time_bins": time_bins
            },
            "consumption_unit": unit,
            "rate_unit": f"dollars / {unit}",
            "sample_rate": None,
            "adjustment_factor": 1.0,
        })
        continue

    # 6) single‐rate
    charges.append({
        "name": f"{comp}_{unit}_single",
        "charge_type": "SingleRateTariff",
        "rate": rate,
        "consumption_unit": unit,
        "rate_unit": f"dollars / {unit}",
        "sample_rate": None,
        "adjustment_factor": 1.0,
    })
    continue

# Emit seasonal BlockTariffs
for (comp, unit, season_txt), sb in seasonal_blocks.items():
    start_dd, end_dd = (s.strip() for s in season_txt.split("-"))
    ms, ds = map(int, start_dd.split("/"))
    me, de = map(int, end_dd.split("/"))

    windows = []
    if (me, de) < (ms, ds):
        windows.append((f"2025-{ms:02d}-{ds:02d}", "2025-12-31"))
        windows.append(("2025-01-01", f"2025-{me:02d}-{de:02d}"))
    else:
        windows.append((f"2025-{ms:02d}-{ds:02d}", f"2025-{me:02d}-{de:02d}"))

    if not sb.get("time_bins"):
        for ws, we in windows:
            # BlockTariff
            charges.append({
                "name": f"{comp}_{unit}_{ws[-5:]}_to_{we[-5:]}_block",
                "charge_type": "BlockTariff",
                "frequency_applied": "month",
                "blocks": sb["blocks"],
                "bin_rates": sb["rates"],
                "bin_labels": sb["labels"],
                "consumption_unit": unit,
                "rate_unit": f"dollars / {unit}",
                "sample_rate": None,
                "adjustment_factor": 1.0,
                "__season_window__": (ws, we)
            })
    else:
        # TOU Block Tariff
        time_windows = list(zip(sb["time_bins"],sb["time_bins"][1:],sb["rates"][0]))
        for ws,we in windows:
            for ts, te, rate in time_windows:
                charges.append({
                    "name": f"{comp}_{unit}_{ws[-5:]}_to_{we[-5:]}_block",
                    "charge_type": "BlockTariff",
                    "frequency_applied": "month",
                    "blocks": sb["blocks"],
                    "bin_rates": [rate],
                    "bin_labels": sb["labels"],
                    "consumption_unit": unit,
                    "rate_unit": f"dollars / {unit}",
                    "sample_rate": None,
                    "adjustment_factor": 1.0,
                    "__season_window__": (ws, we),
                    "__time_window__": (ts, te)
                })


# Emit global BlockTariffs
for (comp, unit), gb in global_blocks.items():
    if not gb.get("time_window"):
        charges.append({
            "name": f"{comp}_{unit}_block",
            "charge_type": "BlockTariff",
            "frequency_applied": "month",
            "blocks": gb["blocks"],
            "bin_rates": gb["rates"],
            "bin_labels": gb["labels"],
            "consumption_unit": unit,
            "rate_unit": f"dollars / {unit}",
            "sample_rate": None,
            "adjustment_factor": 1.0,
        })
    else:
        # TOU Block Tariff
        time_windows = list(zip(sb["time_bins"],sb["time_bins"][1:],gb["rates"][0]))
        for ts, te, rate in time_windows:
            charges.append({
                "name": f"{comp}_{unit}_block",
                "charge_type": "BlockTariff",
                "frequency_applied": "month",
                "blocks": gb["blocks"],
                "bin_rates": [rate],
                "bin_labels": gb["labels"],
                "consumption_unit": unit,
                "rate_unit": f"dollars / {unit}",
                "sample_rate": None,
                "adjustment_factor": 1.0,
                "__time_window__": (ts, te)
            })

tariff=charges

In [187]:

ts = pd.Series(building["electricity.total"].to_numpy(), index=pd.to_datetime(building["timestamp"].to_list()))

# Define the sample rate (hourly)
sample_rate = SampleRate(multiplier=1, base_freq="hours")

# Create the MeterData object
orig_meter = MeterData(
    name="electricity",
    tseries=ts,
    sample_rate=sample_rate,
    units="kwh"
)

# tariff generated from get_tariff_RA
regime_dict = {
    'name': "residential_tariff_regime",
    'tariffs': tariff
}

applied_charges = []
for t in regime_dict['tariffs']:
    meter = orig_meter
    season = t.pop("__season_window__", None)
    time_range = t.pop("__time_window__", None)
    tariff_obj = tariffs_map[t['charge_type']].from_dict(t)
    if season or time_range:
        if season:
            window = DateWindow(start=season[0], end=season[1])
            sliced = meter.window_slice(window)
            meter = MeterData(
                name=meter.name,
                tseries=sliced,
                sample_rate=meter.sample_rate,
                units=meter.units
            )
        if time_range:
            time_window = TimeWindow(start=time_range[0],end = 0 if time_range[1]==24 else time_range[1])
            sliced = meter.window_slice(time_window)
            meter = MeterData(
                name=meter.name,
                tseries=sliced,
                sample_rate=meter.sample_rate,
                units=meter.units
            )
    
    applied_charges.append(tariff_obj.apply(meter))
    
bill = Bill(name="my_combined_bill", charges=applied_charges)

c:\Users\al.qarooni\OneDrive - RMI\Documents\VSCode\.venv\Lib\site-packages\ts_tariffs\tariffs.py:90: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'YE' instead.
  cost_ts = consumption.tseries.resample(


In [188]:
tariff_obj

BlockTariff(name='Conservation Incentive Adjustment - Winter - Territory S_kwh_01-01_to_05-31_block', charge_type='BlockTariff', consumption_unit='kwh', rate_unit='dollars / kwh', sample_rate=None, adjustment_factor=1.0, frequency_applied='month', blocks=[Block(min=0.0, max=10.2), Block(min=10.2, max=inf)], bin_rates=[-0.03568, 0.06733], bin_labels=['', ''])

In [190]:
from collections import defaultdict
import re
print(bill.total)
sums = defaultdict(float)
for k, v in bill.itemised_as_dict.items():
    match = re.match(r"^(.*)(_kwh_|_month_)(\d)", k)
    if match:
        prefix = match.group(1)
        sums[prefix] += v
    else:
        match = re.match(r"^(.*)(_kwh_.*|_month_.*|_year_.*)", k)
        sums[match.group(1)]+=v
    
sums = dict(sorted(sums.items()))
sums

4528.6403700500005


{'California Climate Credit': -116.46,
 'Conservation Incentive Adjustment - Summer - Territory S': 257.11289368999996,
 'Conservation Incentive Adjustment - Winter - Territory S': 349.35105973,
 'Distribution Charge - Summer Off-Peak': 615.0606042299999,
 'Distribution Charge - Summer Peak': 277.24054445999997,
 'Distribution Charge - Winter Off-Peak': 723.8728230600005,
 'Distribution Charge - Winter Peak': 263.17912466999996,
 'Energy Cost Recovery Amount': 0.09305214999999979,
 'Energy Surcharge': 2.7915645000000113,
 'Generation Charge - Summer Off-Peak': 364.33896269999946,
 'Generation Charge - Summer Peak': 268.51135140000025,
 'Generation Charge - Winter Off-Peak': 537.5506442000006,
 'Generation Charge - Winter Peak': 229.29766146000009,
 'New System Generation': 53.41193409999991,
 'Nuclear Decommissioning': -2.2332515999999965,
 'Ongoing CTC': -6.699754800000005,
 'Public Purpose Programs': 246.02988459999875,
 'Recovery Bond Charge': 60.20474104999992,
 'Recovery Bond Cred

In [ ]:
print(costs_breakdown["cost"].sum())
costs_breakdown = costs_breakdown.group_by("rateName").agg([pl.col("rateAmount").mean(), pl.col("cost").sum(),pl.col("itemQuantity").sum().round(3)]).sort(by="rateName")
costs_breakdown.to_dicts()

1495.28338812


[{'rateName': 'Affordability Surcharge',
  'rateAmount': 1.95670389,
  'cost': 25.43715054,
  'itemQuantity': 13.0},
 {'rateName': 'Conservation Improvement Program Adjustment',
  'rateAmount': -0.000389,
  'cost': -3.61931085,
  'itemQuantity': 9304.141},
 {'rateName': 'Customer Charge',
  'rateAmount': 5.99932796,
  'cost': 71.99193548,
  'itemQuantity': 12.0},
 {'rateName': 'Energy Charge - Summer Mid-Peak',
  'rateAmount': 0.11307,
  'cost': 254.22308283,
  'itemQuantity': 2248.369},
 {'rateName': 'Energy Charge - Summer Off-Peak',
  'rateAmount': 0.03825,
  'cost': 38.19120975,
  'itemQuantity': 998.463},
 {'rateName': 'Energy Charge - Summer On-Peak',
  'rateAmount': 0.27845,
  'cost': 193.2804985,
  'itemQuantity': 694.13},
 {'rateName': 'Energy Charge - Winter Mid-Peak',
  'rateAmount': 0.09907,
  'cost': 310.54313781,
  'itemQuantity': 3134.583},
 {'rateName': 'Energy Charge - Winter Off-Peak',
  'rateAmount': 0.03825,
  'cost': 50.94322425,
  'itemQuantity': 1331.849},
 {'rat